# Module 5.1 — Momentum & Trend Following
## Riding Persistence in Prices

---

### On Trends and the Illusion of Randomness

Markets are neither perfectly efficient nor purely random walks at every horizon. **Autocorrelation** in returns is weak at daily frequencies but **persistence** in *direction*—trends—emerges from risk premia, slow-moving capital flows, behavioral underreaction, and institutional constraints. **Momentum and trend-following** strategies formalize a simple bet: what has gone up (down) may continue, until **volatility expansion**, **mean reversion**, or **regime change** ends the move.

This notebook is not a promise of alpha; it is a **map of classical tools** quants use to measure trend, time entries, size risk, and adapt to **regimes**. We pair **moving-average crossovers**, **momentum oscillators** (RSI, MACD), **directional strength** (ADX), **price-channel breakouts** (Donchian / Turtle-style), and **regime-aware** logic. Where the README asks for **RSI mean reversion**, we implement it deliberately as a **counterpoint**: the same indicator serves trend *exhaustion* and short-horizon *reversion* depending on design.

---

### Learning Objectives

1. **Implement** dual moving-average crossover systems and interpret lag vs. noise
2. **Compute** RSI, MACD, and ADX (Wilder-style smoothing) on OHLC data
3. **Build** Donchian / breakout entries and a simplified **Turtle-style** ruleset with ATR sizing
4. **Backtest** long-only trend and **RSI mean-reversion** with turnover costs
5. **Detect** simple **volatility/trend regimes** and compare **regime-conditioned** trend following

---

### Module Contents

1. **Moving averages & crossovers** — The original trend filter
2. **RSI, MACD, ADX** — Strength, momentum, and trend quality
3. **Breakouts & Turtle logic** — Channels and risk-normalized positions
4. **RSI mean reversion** — Oscillator as a different hypothesis
5. **Regime switching** — When trend following lives or dies

---

*"The trend is your friend until the bend at the end."*


In [ ]:
# ========================================
# Setup & synthetic OHLC (multi-regime)
# ========================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from typing import Tuple
from datetime import datetime
import warnings

warnings.filterwarnings("ignore")

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

plt.style.use("seaborn-v0_8-darkgrid")
sns.set_palette("husl")
plt.rcParams.update({
    "figure.figsize": (14, 6),
    "axes.titlesize": 15,
    "figure.dpi": 100,
})

COLORS = {
    "price": "#1565C0",
    "fast": "#E65100",
    "slow": "#6A1B9A",
    "long": "#2E7D32",
    "short": "#C62828",
    "neutral": "#455A64",
    "rsi": "#00838F",
    "regime_hi": "#B71C1C",
    "regime_lo": "#2E7D32",
}


def simulate_ohlc(n_days: int = 2000, seed: int = 42) -> pd.DataFrame:
    """GBM-like close with piecewise drift + vol clusters; build OHLC around close."""
    rng = np.random.default_rng(seed)
    dt = 1 / 252
    mu = np.zeros(n_days)
    sig = np.zeros(n_days)
    # regimes: bull, chop, bear recovery, high vol
    for t in range(n_days):
        if t < 400:
            mu[t], sig[t] = 0.12, 0.14
        elif t < 900:
            mu[t], sig[t] = 0.02, 0.22
        elif t < 1400:
            mu[t], sig[t] = -0.05, 0.18
        else:
            mu[t], sig[t] = 0.08, 0.12
    z = rng.standard_normal(n_days)
    # vol clustering
    h = 0.85
    log_sig = np.zeros(n_days)
    for t in range(1, n_days):
        log_sig[t] = h * log_sig[t - 1] + 0.15 * rng.standard_normal()
    sig = sig * np.exp(log_sig)

    log_c = np.zeros(n_days)
    for t in range(1, n_days):
        log_c[t] = log_c[t - 1] + (mu[t] - 0.5 * sig[t] ** 2) * dt + sig[t] * np.sqrt(dt) * z[t]
    close = 100 * np.exp(log_c)
    # OHLC around close
    w = rng.uniform(0.001, 0.008, n_days)
    high = close * (1 + w)
    low = close * (1 - w)
    open_ = np.r_[close[0], close[:-1]] * (1 + rng.normal(0, 0.002, n_days))
    open_[0] = close[0]
    high = np.maximum(high, np.maximum(open_, close))
    low = np.minimum(low, np.minimum(open_, close))

    idx = pd.bdate_range("2015-01-02", periods=n_days)
    return pd.DataFrame({"open": open_, "high": high, "low": low, "close": close}, index=idx)


ohlc = simulate_ohlc()
print("=" * 78)
print(" " * 14 + "MODULE 5.1: MOMENTUM & TREND FOLLOWING")
print("=" * 78)
print(datetime.now().strftime("%Y-%m-%d %H:%M:%S"))
print(f"OHLC rows: {len(ohlc)} | {ohlc.index[0].date()} -> {ohlc.index[-1].date()}")
print(f"Close ann. vol (full sample): {ohlc['close'].pct_change().std() * np.sqrt(252):.1%}")
ohlc[["close"]].plot(color=COLORS["price"], title="Synthetic close (multi-regime)", figsize=(12, 4))
plt.tight_layout()
plt.show()


---

## 1. Moving Averages & Crossover Systems

### Why quants need this

Moving averages are the **simplest low-pass filters** on price: they reduce noise and define **direction**. Crossover systems encode a **state machine** (long / flat / short) that is easy to audit, explain to risk, and embed in execution. They underperform in chop but remain the **baseline** against which fancier signals are judged.

### Mathematics

For prices $P_t$, an $n$-period **SMA** is $\bar{P}_t = \frac{1}{n}\sum_{i=0}^{n-1} P_{t-i}$. An **EMA** satisfies
$$E_t = \alpha P_t + (1-\alpha)E_{t-1}, \quad \alpha = \frac{2}{n+1}.$$
A **dual MA** goes long when $E^{(\text{fast})}_t > E^{(\text{slow})}_t$ and flat or short otherwise (we use **long-only** below for clarity).


In [ ]:
def ema(series: pd.Series, span: int) -> pd.Series:
    return series.ewm(span=span, adjust=False).mean()


def dual_ma_signal(close: pd.Series, fast: int = 20, slow: int = 60) -> pd.Series:
    ef, es = ema(close, fast), ema(close, slow)
    pos = (ef > es).astype(int)
    return pos, ef, es


def backtest_long_only(
    close: pd.Series,
    position: pd.Series,
    cost_bps: float = 5.0,
) -> Tuple[pd.Series, pd.Series]:
    """Position known at close t applies to return t -> t+1; pay cost on turnover."""
    r = close.pct_change().fillna(0.0)
    pos = position.shift(1).fillna(0.0)
    turn = position.diff().abs().fillna(0.0)
    cost = turn * (cost_bps / 10000.0)
    strat = pos * r - cost
    eq = (1 + strat).cumprod()
    stats_df = pd.Series({
        "Ann_return": strat.mean() * 252,
        "Ann_vol": strat.std() * np.sqrt(252),
        "Sharpe": (strat.mean() * 252 - 0.03) / (strat.std() * np.sqrt(252)) if strat.std() > 0 else np.nan,
        "MaxDD": ((eq / eq.cummax()) - 1).min(),
        "Avg_turnover_day": turn.mean(),
    })
    return strat, stats_df


pos_ma, ef, es = dual_ma_signal(ohlc["close"], 20, 60)
strat_ma, st_ma = backtest_long_only(ohlc["close"], pos_ma)
bh = ohlc["close"].pct_change().fillna(0.0)
eq_bh = (1 + bh).cumprod()
eq_st = (1 + strat_ma).cumprod()

print("Dual MA (20/60) long-only vs buy-hold (same cost model on strategy only):")
print(st_ma.to_string())
print(f"Buy-hold ann. return: {bh.mean()*252:.2%}")

fig, ax = plt.subplots(2, 1, figsize=(14, 8), sharex=True)
ax[0].plot(ohlc.index, ohlc["close"], color=COLORS["price"], lw=1, label="Close")
ax[0].plot(ohlc.index, ef, color=COLORS["fast"], lw=1, label="EMA 20")
ax[0].plot(ohlc.index, es, color=COLORS["slow"], lw=1, label="EMA 60")
ax[0].set_title("Price with fast/slow EMAs")
ax[0].legend(loc="upper left", fontsize=9)
ax[1].plot(eq_bh.index, eq_bh, color=COLORS["neutral"], lw=1, label="Buy & hold")
ax[1].plot(eq_st.index, eq_st, color=COLORS["long"], lw=1.5, label="Dual MA long-only")
ax[1].set_title("Equity curves")
ax[1].legend(fontsize=9)
plt.tight_layout()
plt.show()


---

## 2. RSI, MACD & ADX

### Why quants need this

**RSI** locates **stretch** (potential exhaustion); **MACD** compares **short vs. long-scale trend** of price; **ADX** measures **trend strength** independent of sign. Quants use them for **filters** (trade breakouts only if ADX is high), **timing**, and **risk-off** when trends weaken.

### Definitions (standard)

- **RSI** (Wilder): with average gain $G_t$ and loss $L_t$ over $n$ bars, $RS_t = G_t/L_t$, $RSI_t = 100 - 100/(1+RS_t)$.
- **MACD**: $\text{MACD}_t = E^{(12)}_t - E^{(26)}_t$, signal line $S_t = E^{(9)}(\text{MACD})$, histogram $H_t = \text{MACD}_t - S_t$.
- **ADX**: built from **+DI**, **-DI**, and **true range**; ADX itself is a smoothed magnitude of directional separation (we implement a common recursive form).


In [ ]:
def rsi_wilder(close: pd.Series, n: int = 14) -> pd.Series:
    delta = close.diff()
    gain = delta.clip(lower=0.0)
    loss = (-delta).clip(lower=0.0)
    ag = gain.ewm(alpha=1 / n, adjust=False).mean()
    al = loss.ewm(alpha=1 / n, adjust=False).mean()
    rs = ag / al.replace(0, np.nan)
    out = 100 - (100 / (1 + rs))
    return out.fillna(50.0)


def macd_hist(close: pd.Series, fast: int = 12, slow: int = 26, signal: int = 9) -> pd.DataFrame:
    m = ema(close, fast) - ema(close, slow)
    sig = ema(m, signal)
    hist = m - sig
    return pd.DataFrame({"macd": m, "signal": sig, "hist": hist})


def adx_wilder(high: pd.Series, low: pd.Series, close: pd.Series, n: int = 14) -> pd.DataFrame:
    prev_close = close.shift(1)
    tr = pd.concat([
        high - low,
        (high - prev_close).abs(),
        (low - prev_close).abs(),
    ], axis=1).max(axis=1)

    up = high.diff()
    down = -low.diff()
    plus_dm = ((up > down) & (up > 0)) * up
    minus_dm = ((down > up) & (down > 0)) * down

    atr = tr.ewm(alpha=1 / n, adjust=False).mean()
    plus_di = 100 * (plus_dm.ewm(alpha=1 / n, adjust=False).mean() / atr.replace(0, np.nan))
    minus_di = 100 * (minus_dm.ewm(alpha=1 / n, adjust=False).mean() / atr.replace(0, np.nan))
    dx = (100 * (plus_di - minus_di).abs() / (plus_di + minus_di).replace(0, np.nan)).fillna(0)
    adx = dx.ewm(alpha=1 / n, adjust=False).mean()
    return pd.DataFrame({"plus_di": plus_di, "minus_di": minus_di, "adx": adx, "atr": atr})


r = rsi_wilder(ohlc["close"], 14)
mcdf = macd_hist(ohlc["close"])
adxdf = adx_wilder(ohlc["high"], ohlc["low"], ohlc["close"], 14)

fig, ax = plt.subplots(4, 1, figsize=(14, 11), sharex=True)
ax[0].plot(ohlc.index, ohlc["close"], color=COLORS["price"], lw=1)
ax[0].set_title("Close")
ax[1].plot(r.index, r, color=COLORS["rsi"], lw=1)
ax[1].axhline(70, color="gray", ls="--", lw=0.8)
ax[1].axhline(30, color="gray", ls="--", lw=0.8)
ax[1].set_ylabel("RSI(14)")
ax[1].set_ylim(0, 100)
ax[2].plot(mcdf.index, mcdf["macd"], label="MACD", lw=1)
ax[2].plot(mcdf.index, mcdf["signal"], label="Signal", lw=1)
ax[2].bar(mcdf.index, mcdf["hist"], width=1, alpha=0.35, label="Hist")
ax[2].legend(fontsize=8, ncol=3)
ax[2].set_title("MACD")
ax[3].plot(adxdf.index, adxdf["adx"], color=COLORS["slow"], lw=1, label="ADX")
ax[3].plot(adxdf.index, adxdf["plus_di"], color=COLORS["long"], lw=0.8, alpha=0.8, label="+DI")
ax[3].plot(adxdf.index, adxdf["minus_di"], color=COLORS["short"], lw=0.8, alpha=0.8, label="-DI")
ax[3].set_title("ADX & directional indicators")
ax[3].legend(fontsize=8, ncol=3)
plt.tight_layout()
plt.show()

# Example trend filter: dual MA only when ADX > 25
pos_adx = ((ema(ohlc["close"], 20) > ema(ohlc["close"], 60)) & (adxdf["adx"] > 25)).astype(int)
strat_adx, st_adx = backtest_long_only(ohlc["close"], pos_adx)
print("Dual MA gated by ADX > 25:")
print(st_adx.to_string())


---

## 3. Breakout Strategies & Turtle-Style Rules

### Why quants need this

**Breakouts** trade **range expansion**: a close beyond a **Donchian** high (low) presumes continuation. The **Turtle** experiment showed that simple **channel breakouts** plus **ATR-based position sizing** and **stops** could be traded at scale. Quants need this framework because it separates **entry logic** (price vs. price) from **risk logic** (volatility units).

### Donchian channels

$n$-day high $H^{(n)}_t = \max_{i=1,\ldots,n} P_{t-i}$, low $L^{(n)}_t = \min_{i=1,\ldots,n} P_{t-i}$. A **long entry** on close above prior $H^{(N)}$ is the classic **Turtle System 1** spirit (we use a simplified single system below).

### ATR position sizing (sketch)

**Average True Range** is in `adx_wilder` output. Units risked: stop distance $\approx k \cdot ATR$; position size $\propto \text{risk budget} / (k \cdot ATR)$ (we normalize to long-only 0/1 for the backtest, but plot **notional-scaled** exposure as tutorial).


In [ ]:
def donchian(high: pd.Series, low: pd.Series, n: int) -> pd.DataFrame:
    up = high.shift(1).rolling(n).max()
    dn = low.shift(1).rolling(n).min()
    return pd.DataFrame({"upper": up, "lower": dn})


def turtle_style_signals(close: pd.Series, high: pd.Series, low: pd.Series,
                         entry_n: int = 20, exit_n: int = 10) -> pd.DataFrame:
    d_e = donchian(high, low, entry_n)
    d_x = donchian(high, low, exit_n)
    long_entry = close > d_e["upper"]
    long_exit = close < d_x["lower"]
    pos = np.zeros(len(close), dtype=int)
    state = 0
    for i in range(len(close)):
        if state == 0 and long_entry.iloc[i]:
            state = 1
        elif state == 1 and long_exit.iloc[i]:
            state = 0
        pos[i] = state
    sig = pd.Series(pos, index=close.index)
    return pd.DataFrame({"position": sig, "donchian_hi": d_e["upper"], "donchian_exit": d_x["lower"]})


dc = turtle_style_signals(ohlc["close"], ohlc["high"], ohlc["low"], 20, 10)
strat_tb, st_tb = backtest_long_only(ohlc["close"], dc["position"])
print("Turtle-style Donchian 20 entry / 10 exit (long-only):")
print(st_tb.to_string())

atr = adx_wilder(ohlc["high"], ohlc["low"], ohlc["close"], 14)["atr"]
risk_budget = 0.01  # 1% of equity per day at risk (toy)
notional_scale = (risk_budget / (2 * atr / ohlc["close"])).clip(0, 3)  # toy leverage cap

fig, ax = plt.subplots(3, 1, figsize=(14, 9), sharex=True)
ax[0].plot(ohlc.index, ohlc["close"], color=COLORS["price"], lw=1, label="Close")
ax[0].plot(dc.index, dc["donchian_hi"], color=COLORS["fast"], lw=0.8, alpha=0.9, label="Donchian entry (prev high)")
ax[0].plot(dc.index, dc["donchian_exit"], color=COLORS["short"], lw=0.8, alpha=0.7, label="Exit channel low")
ax[0].set_title("Breakout channels (shifted Donchian)")
ax[0].legend(fontsize=8)
ax[1].fill_between(dc.index, 0, dc["position"], color=COLORS["long"], alpha=0.25, step="post")
ax[1].set_ylim(-0.1, 1.1)
ax[1].set_title("Long-only Turtle-style position")
ax[2].plot(notional_scale.index, notional_scale, color=COLORS["neutral"], lw=1)
ax[2].set_title("Toy ATR-scaled exposure cap (illustrative)")
plt.tight_layout()
plt.show()


---

## 4. RSI Mean Reversion (Counterpoint)

### Why quants need this

The README places **RSI mean reversion** beside trend tools to stress one fact: **the same statistic supports opposite economic stories**. Mean reversion bets on **short-horizon overextension**; trend systems bet on **persistence**. Quants must know **which hypothesis** a rule encodes—and that **parameter stability** differs by regime.

### Rule (illustration)

Go **long** when RSI crosses below 30 and **exit** when RSI crosses above 55 (or flat after `max_hold` days). Long-only, symmetric logic could be added for shorts on RSI > 70.


In [ ]:
def rsi_mean_reversion_positions(close: pd.Series, n: int = 14,
                                  buy_level: float = 30.0, sell_level: float = 55.0,
                                  max_hold: int = 15) -> pd.Series:
    r = rsi_wilder(close, n)
    pos = np.zeros(len(close), dtype=int)
    hold = 0
    for i in range(1, len(close)):
        if pos[i - 1] == 0 and r.iloc[i] < buy_level:
            pos[i] = 1
            hold = 0
        elif pos[i - 1] == 1:
            hold += 1
            if r.iloc[i] > sell_level or hold >= max_hold:
                pos[i] = 0
            else:
                pos[i] = 1
        else:
            pos[i] = 0
    return pd.Series(pos, index=close.index)


pos_rsi = rsi_mean_reversion_positions(ohlc["close"])
strat_rsi, st_rsi = backtest_long_only(ohlc["close"], pos_rsi)
print("RSI mean reversion (long-only, toy thresholds):")
print(st_rsi.to_string())

fig, ax = plt.subplots(2, 1, figsize=(14, 6), sharex=True)
ax[0].plot(ohlc.index, ohlc["close"], color=COLORS["price"], lw=1)
ax[0].set_title("Close")
r = rsi_wilder(ohlc["close"], 14)
ax[1].plot(r.index, r, color=COLORS["rsi"], lw=1)
ax[1].axhline(30, color=COLORS["long"], ls="--", lw=0.8)
ax[1].axhline(55, color=COLORS["fast"], ls="--", lw=0.8)
ax[1].set_ylabel("RSI")
ax[1].set_title("RSI with mean-reversion thresholds")
plt.tight_layout()
plt.show()


---

## 5. Regime Detection & Regime-Switching Trend Following

### Why quants need this

**Trend following earns crisis convexity in some eras and bleeds in choppy mean-reverting tape.** Simple **regime labels**—high vs. low volatility, or **trend strength**—let quants **scale** exposure or **switch** parameter sets (fast vs. slow MA). Full **HMM** models are common in production; here we use **transparent rules** you can audit.

### Rule-based regimes

- **Vol regime**: compare short rolling volatility of returns to its **long median** (high if above 75th percentile of expanding history).
- **Strategy**: use **fast MA (10/40)** in high-vol regimes and **slow MA (30/120)** in low-vol regimes (illustration only).


In [ ]:
def vol_regime(close: pd.Series, short: int = 20, quantile: float = 0.75) -> pd.Series:
    r = close.pct_change()
    rv = r.rolling(short).std()
    # expanding quantile threshold to avoid look-ahead in live use, use past only
    thr = rv.expanding(min_periods=120).quantile(quantile)
    return (rv > thr).astype(int)


reg = vol_regime(ohlc["close"])
fast_sig, _, _ = dual_ma_signal(ohlc["close"], 10, 40)
slow_sig, _, _ = dual_ma_signal(ohlc["close"], 30, 120)
pos_reg = np.where(reg == 1, fast_sig, slow_sig)
pos_reg = pd.Series(pos_reg, index=ohlc.index)

strat_reg, st_reg = backtest_long_only(ohlc["close"], pos_reg)
strat_slow, st_slow = backtest_long_only(ohlc["close"], slow_sig)
strat_fast, st_fast = backtest_long_only(ohlc["close"], fast_sig)

summary = pd.DataFrame({
    "Dual MA 20/60": st_ma,
    "ADX-gated MA": st_adx,
    "Turtle-style": st_tb,
    "RSI MR": st_rsi,
    "MA 10/40": st_fast,
    "MA 30/120": st_slow,
    "Regime-switch MA": st_reg,
})
print("Strategy comparison (same backtest machinery, 5 bps per |Δw|):")
print(summary.T.to_string())

fig, ax = plt.subplots(3, 1, figsize=(14, 9), sharex=True)
ax[0].plot(ohlc.index, ohlc["close"], color=COLORS["price"], lw=1)
ax[0].set_title("Close")
ax[1].fill_between(reg.index, 0, reg, color=COLORS["regime_hi"], alpha=0.25, step="post", label="High vol regime")
ax[1].set_ylim(-0.1, 1.1)
ax[1].set_title("Volatility regime (1 = high short-term vol)")
ax[1].legend(fontsize=8)
cum = (1 + backtest_long_only(ohlc["close"], pos_reg)[0]).cumprod()
cum_s = (1 + backtest_long_only(ohlc["close"], slow_sig)[0]).cumprod()
ax[2].plot(cum.index, cum, color=COLORS["long"], lw=1.5, label="Regime-switch MA")
ax[2].plot(cum_s.index, cum_s, color=COLORS["neutral"], lw=1, alpha=0.8, label="Slow MA only")
ax[2].set_title("Equity: regime-switch vs. slow MA")
ax[2].legend(fontsize=9)
plt.tight_layout()
plt.show()

print("\n" + "=" * 78)
print("MODULE 5.1 — SUMMARY")
print("=" * 78)
print("Trend tools encode persistence; oscillators encode stretch; ADX encodes strength.")
print("Breakouts channel expansion; Turtles pair channels with ATR risk.")
print("RSI mean reversion is a different hypothesis—regimes decide who suffers.")
print("=" * 78)
